<a href="https://colab.research.google.com/github/njalagam23/neural-networks-from-scratch/blob/main/day09_automated_testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/njalagam23/neural-networks-from-scratch.git /content/churn-repo

Cloning into '/content/churn-repo'...
remote: Enumerating objects: 68, done.
remote: Counting objects: 100% (68/68), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 68 (delta 21), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (68/68), 255.22 KiB | 1.39 MiB/s, done.
Resolving deltas: 100% (21/21), done.


In [2]:
from pathlib import Path

project = Path("/content/churn-repo/customer-churn-app")

print("Application found:", project.exists())
print("Prediction module found:", (project / "app/predict.py").exists())
print("Model weights found:", (project / "artifacts/churn_model.pth").exists())

Application found: True
Prediction module found: True
Model weights found: True


In [5]:
import sys

sys.path.insert(0, "/content/churn-repo/customer-churn-app")

from app.predict import ChurnPredictor

predictor = ChurnPredictor()

print("Model loaded successfully!")

Model loaded successfully!


In [6]:
result = predictor.predict(
    usage=5,
    tenure=8,
    tickets=3
)

print(result)

{'churn_probability': 0.7995, 'risk_level': 'High Churn Risk'}


In [7]:
!cd /content/churn-repo/customer-churn-app && python -m pytest tests/test_prediction.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/churn-repo/customer-churn-app
plugins: platformdirs-4.12.2, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.2
collected 0 items                                                              

============================ no tests ran in 0.01s =============================
ERROR: file or directory not found: tests/test_prediction.py



In [8]:
from pathlib import Path

project = Path("/content/churn-repo/customer-churn-app")
tests_folder = project / "tests"

tests_folder.mkdir(parents=True, exist_ok=True)

print("Tests folder exists:", tests_folder.exists())

Tests folder exists: True


In [9]:
%%writefile /content/churn-repo/customer-churn-app/tests/test_prediction.py

from app.predict import ChurnPredictor

def test_prediction_probability():
    predictor = ChurnPredictor()

    result = predictor.predict(
        usage=5,
        tenure=8,
        tickets=3
    )

    assert 0 <= result["churn_probability"] <= 1

Writing /content/churn-repo/customer-churn-app/tests/test_prediction.py


In [10]:
test_file = project / "tests/test_prediction.py"

print("Test file exists:", test_file.exists())

Test file exists: True


In [11]:
!cd /content/churn-repo/customer-churn-app && python -m pytest tests/test_prediction.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/churn-repo/customer-churn-app
plugins: platformdirs-4.12.2, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.2
collected 1 item                                                               

tests/test_prediction.py::test_prediction_probability PASSED             [100%]

============================== 1 passed in 2.86s ===============================


In [12]:
%%writefile /content/churn-repo/customer-churn-app/tests/test_prediction.py

import pytest

from app.predict import ChurnPredictor


@pytest.fixture(scope="module")
def predictor():
    return ChurnPredictor()


def test_prediction_probability(predictor):
    result = predictor.predict(
        usage=5,
        tenure=8,
        tickets=3
    )

    assert 0 <= result["churn_probability"] <= 1


@pytest.mark.parametrize(
    "usage, tenure, tickets, expected_risk",
    [
        (5, 8, 3, "High Churn Risk"),
        (30, 48, 0, "Low Churn Risk"),
        (12, 15, 2, "High Churn Risk"),
    ]
)
def test_customer_risk(
    predictor, usage, tenure, tickets, expected_risk
):
    result = predictor.predict(usage, tenure, tickets)

    assert result["risk_level"] == expected_risk

Overwriting /content/churn-repo/customer-churn-app/tests/test_prediction.py


In [13]:
!cd /content/churn-repo/customer-churn-app && python -m pytest tests/test_prediction.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/churn-repo/customer-churn-app
plugins: platformdirs-4.12.2, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.2
collected 4 items                                                              

tests/test_prediction.py::test_prediction_probability PASSED             [ 25%]
tests/test_prediction.py::test_customer_risk[5-8-3-High Churn Risk] PASSED [ 50%]
tests/test_prediction.py::test_customer_risk[30-48-0-Low Churn Risk] PASSED [ 75%]
tests/test_prediction.py::test_customer_risk[12-15-2-High Churn Risk] PASSED [100%]

============================== 4 passed in 2.70s ===============================


In [15]:
%%writefile -a /content/churn-repo/customer-churn-app/tests/test_prediction.py


@pytest.mark.parametrize(
    "usage, tenure, tickets",
    [
        (-5, 8, 3),
        ("unknown", 8, 3),
        (5, -2, 3),
        (5, 8, -1),
    ]
)
def test_invalid_customer_inputs(
    predictor, usage, tenure, tickets
):
    with pytest.raises(ValueError):
        predictor.predict(usage, tenure, tickets)

Appending to /content/churn-repo/customer-churn-app/tests/test_prediction.py


In [16]:
!cd /content/churn-repo/customer-churn-app && python -m pytest tests/test_prediction.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/churn-repo/customer-churn-app
plugins: platformdirs-4.12.2, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.2
collected 8 items                                                              

tests/test_prediction.py::test_prediction_probability PASSED             [ 12%]
tests/test_prediction.py::test_customer_risk[5-8-3-High Churn Risk] PASSED [ 25%]
tests/test_prediction.py::test_customer_risk[30-48-0-Low Churn Risk] PASSED [ 37%]
tests/test_prediction.py::test_customer_risk[12-15-2-High Churn Risk] PASSED [ 50%]
tests/test_prediction.py::test_invalid_customer_inputs[-5-8-3] PASSED    [ 62%]
tests/test_prediction.py::test_invalid_customer_inputs[unknown-8-3] PASSED [ 75%]
tests/test_prediction.py::test_invalid_customer_inputs[5--2-3] PASSED    [ 87%]
tests/test_prediction.py::test_inval

In [17]:
%%writefile -a /content/churn-repo/customer-churn-app/tests/test_prediction.py


def test_model_loading(predictor):

    # The model should be in evaluation mode
    assert predictor.model.training is False

    # Verify model architecture
    assert predictor.model.hidden.in_features == 3
    assert predictor.model.output.out_features == 1


def test_prediction_consistency(predictor):

    # Predict twice for the same customer
    result1 = predictor.predict(5, 8, 3)
    result2 = predictor.predict(5, 8, 3)

    # Results should match
    assert result1 == result2

Appending to /content/churn-repo/customer-churn-app/tests/test_prediction.py


In [18]:
!cd /content/churn-repo/customer-churn-app && python -m pytest tests/ -v

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/churn-repo/customer-churn-app
plugins: platformdirs-4.12.2, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.2
collected 10 items                                                             

tests/test_prediction.py::test_prediction_probability PASSED             [ 10%]
tests/test_prediction.py::test_customer_risk[5-8-3-High Churn Risk] PASSED [ 20%]
tests/test_prediction.py::test_customer_risk[30-48-0-Low Churn Risk] PASSED [ 30%]
tests/test_prediction.py::test_customer_risk[12-15-2-High Churn Risk] PASSED [ 40%]
tests/test_prediction.py::test_invalid_customer_inputs[-5-8-3] PASSED    [ 50%]
tests/test_prediction.py::test_invalid_customer_inputs[unknown-8-3] PASSED [ 60%]
tests/test_prediction.py::test_invalid_customer_inputs[5--2-3] PASSED    [ 70%]
tests/test_prediction.py::test_inval

In [19]:
from google.colab import files

files.download(
    "/content/churn-repo/customer-churn-app/tests/test_prediction.py"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>